# Symbolic + Neural Hybrid Generator

This notebook demonstrates a hybrid system where symbolic rules define high-level structures or constraints, and a neural component realizes fluent natural language output. Inspired by neurosymbolic AI (e.g., combining logic from symbolic planners with neural generation ala GPT).

## Overview
- **Symbolic Part**: Use a rule-based system to generate structured templates (e.g., via simple logic or SymPy for symbolic manipulation).
- **Neural Part**: Use a basic PyTorch RNN to 'realize' or fluent-ify the output.
- Example Domain: Generating simple educational sentences with factual constraints.

We'll simulate 'GPT realization' with a toy neural language model trained on a small corpus.

In [ ]:
import sympy as sp  # For symbolic rules
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import Dataset, DataLoader

# Step 1: Symbolic Rules (e.g., constraint satisfaction)
def symbolic_rule_generator(facts):
    # Simple rule: Combine facts with logical AND/OR
    x, y = sp.symbols('x y')
    rule = sp.And(x, y)  # Symbolic representation
    template = f"{facts[0]} and {facts[1]} implies {sp.simplify(rule.subs({x: facts[0], y: facts[1]}))} "
    return template

# Example facts
facts = ["Earth is round", "Water boils at 100°C"]
symbolic_template = symbolic_rule_generator(facts)
print("Symbolic Template:", symbolic_template)

## Neural Realization Component

Here, we train a simple RNN to generate fluent text from a seed (the symbolic template). This mimics 'GPT realization' but on a tiny scale.

In [ ]:
# Toy dataset for neural fluency (educational sentences)
corpus = [
    "The Earth is round and orbits the sun.",
    "Water boils at 100 degrees Celsius under standard pressure.",
    "Photosynthesis converts light into energy for plants."
]

# Vocabulary
vocab = list(set(' '.join(corpus).split()))
vocab_size = len(vocab)
word_to_idx = {w: i for i, w in enumerate(vocab)}
idx_to_word = {i: w for w, i in word_to_idx.items()}

class TextDataset(Dataset):
    def __init__(self, corpus):
        self.data = [torch.tensor([word_to_idx[w] for w in sentence.split()]) for sentence in corpus]
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        seq = self.data[idx]
        return seq[:-1], seq[1:]

dataset = TextDataset(corpus)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

# Simple RNN Model
class RNNGenerator(nn.Module):
    def __init__(self, vocab_size, embed_size=10, hidden_size=20):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.RNN(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)
    
    def forward(self, x, hidden=None):
        x = self.embedding(x)
        out, hidden = self.rnn(x, hidden)
        out = self.fc(out)
        return out, hidden

model = RNNGenerator(vocab_size)
optimizer = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss()

# Train (toy training loop)
for epoch in range(10):
    for inputs, targets in dataloader:
        optimizer.zero_grad()
        outputs, _ = model(inputs)
        loss = criterion(outputs.view(-1, vocab_size), targets.view(-1))
        loss.backward()
        optimizer.step()
print("Model trained on toy data.")

In [ ]:
# Hybrid Generation: Rules → Neural Realization
def hybrid_generate(template):
    # Seed with template words
    seed_words = template.split()[:3]  # Take first few words
    seed_idx = torch.tensor([[word_to_idx.get(w, 0) for w in seed_words]])
    
    model.eval()
    generated = seed_words.copy()
    hidden = None
    for _ in range(10):  # Generate 10 more words
        out, hidden = model(seed_idx, hidden)
        next_word_idx = torch.argmax(out[0, -1]).item()
        generated.append(idx_to_word[next_word_idx])
        seed_idx = torch.tensor([[next_word_idx]])
    
    return ' '.join(generated)

output = hybrid_generate(symbolic_template)
print("Hybrid Output:", output)

## Discussion
This hybrid approach ensures rule-based accuracy (symbolic) with fluent generation (neural). Extend by integrating reward feedback for constraint satisfaction, as per your learned topics.